In [2]:
import pandas as pd
import time
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup

session = Session()

bucket = session.default_bucket()

print(bucket)

sagemaker-us-east-1-006228610000


In [3]:
df = pd.read_csv("Dataset/weatherHistory.csv")

In [4]:
df = df[
    [
        "Humidity",
        "Wind Speed (km/h)",
        "Wind Bearing (degrees)",
        "Visibility (km)",
        "Pressure (millibars)",
        "Temperature (C)"
    ]
]

In [5]:
df["record_id"] = range(len(df))

/tmp/ipykernel_1346/837867430.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["record_id"] = range(len(df))


In [6]:
df.head()

,Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars),Temperature (C),record_id
0,0.89,14.1197,251.0,15.8263,1015.13,9.472222,0
1,0.86,14.2646,259.0,15.8263,1015.63,9.355556,1
2,0.89,3.9284,204.0,14.9569,1015.94,9.377778,2
3,0.83,14.1036,269.0,15.8263,1016.41,8.288889,3
4,0.83,11.0446,259.0,15.8263,1016.51,8.755556,4


In [7]:
df["event_time"] = pd.Timestamp.now().strftime("%Y-%m-%dT%H:%M:%SZ")

In [8]:
df.head()

,Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars),Temperature (C),record_id,event_time
0,0.89,14.1197,251.0,15.8263,1015.13,9.472222,0,2026-06-19T19:41:59Z
1,0.86,14.2646,259.0,15.8263,1015.63,9.355556,1,2026-06-19T19:41:59Z
2,0.89,3.9284,204.0,14.9569,1015.94,9.377778,2,2026-06-19T19:41:59Z
3,0.83,14.1036,269.0,15.8263,1016.41,8.288889,3,2026-06-19T19:41:59Z
4,0.83,11.0446,259.0,15.8263,1016.51,8.755556,4,2026-06-19T19:41:59Z


In [9]:
import sagemaker

role = sagemaker.get_execution_role()

print(role)

arn:aws:iam::006228610000:role/LabRole


In [16]:
feature_group = FeatureGroup(
    name="weather-feature-group-v2",
    sagemaker_session=session
)

In [17]:
feature_group.load_feature_definitions(data_frame=df)

[FeatureDefinition(feature_name='humidity', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='wind_speed', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='wind_bearing', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='visibility', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='pressure', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='temperature', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='record_id', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='event_time', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None)]

In [18]:
df = df.rename(columns={
    "Wind Speed (km/h)": "wind_speed",
    "Wind Bearing (degrees)": "wind_bearing",
    "Visibility (km)": "visibility",
    "Pressure (millibars)": "pressure",
    "Temperature (C)": "temperature",
    "Humidity": "humidity"
})

In [19]:
print(df.columns)

Index(['humidity', 'wind_speed', 'wind_bearing', 'visibility', 'pressure',
       'temperature', 'record_id', 'event_time'],
      dtype='object')


In [20]:
feature_group.create(
    s3_uri=f"s3://{bucket}/feature-store",
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True
)

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:006228610000:feature-group/weather-feature-group-v2',
 'ResponseMetadata': {'RequestId': '084752ee-83f8-46b1-9cf2-a3062e2ba822',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '084752ee-83f8-46b1-9cf2-a3062e2ba822',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '101',
   'date': 'Fri, 19 Jun 2026 19:45:43 GMT'},
  'RetryAttempts': 0}}

In [22]:
feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:006228610000:feature-group/weather-feature-group-v2',
 'FeatureGroupName': 'weather-feature-group-v2',
 'RecordIdentifierFeatureName': 'record_id',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'humidity',
   'FeatureType': 'Fractional'},
  {'FeatureName': 'wind_speed', 'FeatureType': 'Fractional'},
  {'FeatureName': 'wind_bearing', 'FeatureType': 'Fractional'},
  {'FeatureName': 'visibility', 'FeatureType': 'Fractional'},
  {'FeatureName': 'pressure', 'FeatureType': 'Fractional'},
  {'FeatureName': 'temperature', 'FeatureType': 'Fractional'},
  {'FeatureName': 'record_id', 'FeatureType': 'Integral'},
  {'FeatureName': 'event_time', 'FeatureType': 'String'}],
 'CreationTime': datetime.datetime(2026, 6, 19, 19, 45, 42, 892000, tzinfo=tzlocal()),
 'OnlineStoreConfig': {'EnableOnlineStore': True},
 'OfflineStoreConfig': {'S3StorageConfig': {'S3Uri': 's3://sagemaker-us-east-1-006228610000/feature-store',
   '